In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Load data
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# Clean both datasets
for df in [train, test]:
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["Title"] = df["Name"].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df["Title"] = df["Title"].map({
        "Mr": 0, "Miss": 1, "Mrs": 2,
        "Master": 3, "Dr": 4}).fillna(5)

# Features
features = ["Pclass", "Sex", "Age", "Fare",
            "FamilySize", "IsAlone", "Title"]

X = train[features]
y = train["Survived"]
X_test = test[features]

# SVM Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf'))
])

# GridSearch
param_grid = {
    'model__C': [0.1, 1, 10],
    'model__gamma': ['scale', 'auto']
}

grid = GridSearchCV(pipeline, param_grid, cv=5)
grid.fit(X, y)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_.round(3))

# Submit
best_model = grid.best_estimator_
predictions = best_model.predict(X_test)
output = pd.DataFrame({
    'PassengerId': test.PassengerId,
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)
print("Done! Submit now!")

Best Parameters: {'model__C': 1, 'model__gamma': 'scale'}
Best CV Score: 0.829
Done! Submit now!


In [4]:
# Back to basics!
features = ["Pclass", "Sex", "SibSp", "Parch", "Fare"]

X = pd.get_dummies(train[features])
X_test = pd.get_dummies(test[features])

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', C=1, gamma='scale'))
])

pipeline.fit(X, y)
predictions = pipeline.predict(X_test)

output = pd.DataFrame({
    'PassengerId': test.PassengerId,
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)
print("Done!")

Done!
